# บทที่ 9: การประเมินผลโมเดล (Model Evaluation)

ใน Notebook นี้ เราจะเรียนรู้ Metrics ต่างๆ สำหรับประเมินผลโมเดล ได้แก่ Accuracy, Precision, Recall, F1-Score, ROC Curve และ Cross-Validation

## 1. นำเข้าไลบรารี

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import seaborn as sns

# ติดตั้งฟอนต์ภาษาไทยสำหรับ Google Colab
import subprocess, glob
subprocess.run(['apt-get', 'install', '-y', '-qq', 'fonts-tlwg-garuda'], 
               capture_output=True)

# ลงทะเบียนฟอนต์โดยตรง
from matplotlib.font_manager import fontManager
for font_file in glob.glob('/usr/share/fonts/truetype/tlwg/*.ttf'):
    fontManager.addfont(font_file)

# ตั้งค่า Seaborn theme และฟอนต์ภาษาไทย
sns.set_theme(style='whitegrid', font='Garuda')
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['figure.figsize'] = (10, 6)
%config InlineBackend.figure_format = 'retina'

from sklearn.metrics import (confusion_matrix, accuracy_score, precision_score, 
                             recall_score, f1_score, roc_curve, auc,
                             classification_report)

np.random.seed(42)

## 2. Confusion Matrix

In [ ]:
# สร้างข้อมูลตัวอย่าง
y_true = np.array([0, 0, 0, 0, 1, 1, 1, 1, 1, 1])
y_pred = np.array([0, 0, 1, 0, 1, 1, 0, 1, 1, 1])

# Confusion Matrix
cm = confusion_matrix(y_true, y_pred)

print("=== Confusion Matrix ===")
print(cm)
print("\nLayout:")
print("         Predicted")
print("         Neg  Pos")
print("Actual Neg  TN   FP")
print("       Pos  FN   TP")

# Extract values
tn, fp, fn, tp = cm.ravel()
print(f"\nTN (True Negative): {tn}")
print(f"FP (False Positive): {fp}")
print(f"FN (False Negative): {fn}")
print(f"TP (True Positive): {tp}")

## 3. Classification Metrics

In [ ]:
def calculate_metrics(y_true, y_pred):
    """
    Calculate classification metrics from scratch
    """
    cm = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()
    
    # Accuracy = (TP + TN) / (TP + TN + FP + FN)
    accuracy = (tp + tn) / (tp + tn + fp + fn)
    
    # Precision = TP / (TP + FP)
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    
    # Recall (Sensitivity) = TP / (TP + FN)
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    
    # F1-Score = 2 × (Precision × Recall) / (Precision + Recall)
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    
    # Specificity = TN / (TN + FP)
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    
    return {
        'Accuracy': accuracy,
        'Precision': precision,
        'Recall': recall,
        'F1-Score': f1,
        'Specificity': specificity
    }

metrics = calculate_metrics(y_true, y_pred)

print("=== Classification Metrics ===")
for name, value in metrics.items():
    print(f"{name}: {value:.4f}")

## 4. ROC Curve และ AUC

In [ ]:
# สร้างข้อมูลตัวอย่างพร้อม probabilities
y_true_binary = np.array([0, 0, 1, 0, 1, 1, 0, 1, 1, 1])
y_scores = np.array([0.1, 0.3, 0.8, 0.2, 0.6, 0.9, 0.4, 0.7, 0.85, 0.95])

# Calculate ROC curve
fpr, tpr, thresholds = roc_curve(y_true_binary, y_scores)
roc_auc = auc(fpr, tpr)

# Plot
plt.figure(figsize=(10, 6))
plt.plot(fpr, tpr, 'b-', linewidth=2, label=f'ROC Curve (AUC = {roc_auc:.4f})')
plt.plot([0, 1], [0, 1], 'r--', label='Random Classifier')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate (Recall)')
plt.title('ROC Curve')
plt.legend()
plt.show()

print(f"AUC: {roc_auc:.4f}")

## 5. Precision-Recall Trade-off

In [ ]:
# แสดง Precision-Recall Trade-off
def precision_recall_at_threshold(y_true, y_scores, threshold):
    """Calculate precision and recall at a given threshold"""
    y_pred = (y_scores >= threshold).astype(int)
    metrics = calculate_metrics(y_true, y_pred)
    return metrics['Precision'], metrics['Recall']

thresholds_to_test = [0.3, 0.5, 0.7, 0.9]

print("=== Precision-Recall at Different Thresholds ===")
print(f"{'Threshold':<12} {'Precision':<12} {'Recall':<12}")
print("-" * 36)

for thresh in thresholds_to_test:
    prec, rec = precision_recall_at_threshold(y_true_binary, y_scores, thresh)
    print(f"{thresh:<12.2f} {prec:<12.4f} {rec:<12.4f}")

## 6. Regression Metrics

In [ ]:
def calculate_regression_metrics(y_true, y_pred):
    """Calculate regression metrics"""
    # Mean Absolute Error (MAE)
    mae = np.mean(np.abs(y_true - y_pred))
    
    # Mean Squared Error (MSE)
    mse = np.mean((y_true - y_pred)**2)
    
    # Root Mean Squared Error (RMSE)
    rmse = np.sqrt(mse)
    
    # R² Score
    ss_res = np.sum((y_true - y_pred)**2)
    ss_tot = np.sum((y_true - np.mean(y_true))**2)
    r2 = 1 - (ss_res / ss_tot)
    
    return {
        'MAE': mae,
        'MSE': mse,
        'RMSE': rmse,
        'R²': r2
    }

# Example
y_true_reg = np.array([100, 150, 200, 250, 300])
y_pred_reg = np.array([110, 145, 210, 240, 290])

reg_metrics = calculate_regression_metrics(y_true_reg, y_pred_reg)

print("=== Regression Metrics ===")
for name, value in reg_metrics.items():
    print(f"{name}: {value:.4f}")

## 7. K-Fold Cross-Validation

In [ ]:
def k_fold_split(n_samples, k=5, shuffle=True):
    """Generate K-fold cross-validation indices"""
    indices = np.arange(n_samples)
    if shuffle:
        np.random.shuffle(indices)
        
    fold_size = n_samples // k
    folds = []
    
    for i in range(k):
        start = i * fold_size
        if i == k - 1:
            end = n_samples  # Last fold takes remaining samples
        else:
            end = start + fold_size
            
        val_indices = indices[start:end]
        train_indices = np.concatenate([indices[:start], indices[end:]])
        folds.append((train_indices, val_indices))
        
    return folds

# Example
n_samples = 100
k = 5
folds = k_fold_split(n_samples, k)

print(f"=== {k}-Fold Cross-Validation ===")
print(f"Total samples: {n_samples}")
print(f"\nFold sizes:")
for i, (train_idx, val_idx) in enumerate(folds):
    print(f"Fold {i+1}: Train={len(train_idx)}, Val={len(val_idx)}")

## 8. แบบฝึกหัดการคำนวณ

### แบบฝึกหัดที่ 1: คำนวณ Metrics

In [ ]:
# ให้ Confusion Matrix: TN=80, FP=10, FN=5, TP=50
# จงคำนวณ Accuracy, Precision, Recall, F1-Score

tn, fp, fn, tp = 80, 10, 5, 50

accuracy = (tp + tn) / (tp + tn + fp + fn)
precision = tp / (tp + fp)
recall = tp / (tp + fn)
f1 = 2 * precision * recall / (precision + recall)

print(f"Accuracy: {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall:.4f}")
print(f"F1-Score: {f1:.4f}")

### แบบฝึกหัดที่ 2: Regression Metrics

In [ ]:
# ให้ y_true = [3, 5, 7, 9, 11] และ y_pred = [2.5, 5.5, 6.8, 9.2, 10.5]
# จงคำนวณ MAE, MSE, RMSE

y_true = np.array([3, 5, 7, 9, 11])
y_pred = np.array([2.5, 5.5, 6.8, 9.2, 10.5])

mae = np.mean(np.abs(y_true - y_pred))
mse = np.mean((y_true - y_pred)**2)
rmse = np.sqrt(mse)

print(f"MAE: {mae:.4f}")
print(f"MSE: {mse:.4f}")
print(f"RMSE: {rmse:.4f}")

### แบบฝึกหัดที่ 3: Threshold Effect

In [ ]:
# ให้ probabilities = [0.2, 0.4, 0.6, 0.8] และ true labels = [0, 0, 1, 1]
# จงคำนวณ accuracy ที่ threshold = 0.5 และ threshold = 0.7

probabilities = np.array([0.2, 0.4, 0.6, 0.8])
true_labels = np.array([0, 0, 1, 1])

# Threshold = 0.5
pred_05 = (probabilities >= 0.5).astype(int)
acc_05 = np.mean(pred_05 == true_labels)

# Threshold = 0.7
pred_07 = (probabilities >= 0.7).astype(int)
acc_07 = np.mean(pred_07 == true_labels)

print(f"Threshold 0.5: predictions={pred_05}, accuracy={acc_05:.4f}")
print(f"Threshold 0.7: predictions={pred_07}, accuracy={acc_07:.4f}")

### แบบฝึกหัดที่ 4: F1-Score Trade-off

In [ ]:
# ให้ Precision = 0.8 และ Recall = 0.6
# จงคำนวณ F1-Score

precision = 0.8
recall = 0.6

f1 = 2 * precision * recall / (precision + recall)
print(f"Precision: {precision}")
print(f"Recall: {recall}")
print(f"F1-Score: {f1:.4f}")

## บทสรุป

Notebook นี้ครอบคลุม:
1. **Confusion Matrix**: TN, FP, FN, TP
2. **Classification Metrics**: Accuracy, Precision, Recall, F1-Score
3. **ROC Curve & AUC**: ประเมินโมเดลที่ output เป็น probability
4. **Regression Metrics**: MAE, MSE, RMSE, R²
5. **Cross-Validation**: K-Fold